[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: artefactos en GitHub Actions

*Nota para quien imparte el curso: el cuaderno 11 no incluye una actividad con enunciado. Los ejercicios de este cuaderno se derivan de su texto («es posible acceder y descargar el contenido de `cobertura-pytest`») y se listan en el informe de revisión.*

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Reproducir en local el informe de cobertura

El *workflow* `ci.yml` ejecuta *pytest* y publica `api/coverage.xml` como el artefacto `cobertura-pytest`. Generar ese archivo en local y comprobar que cumple la cobertura mínima configurada.

La siguiente celda copiará el repositorio a una carpeta temporal, creará un entorno virtual con las dependencias de desarrollo (requiere acceso a *PyPI*) y ejecutará *pytest* en el directorio `api`, igual que el paso `Pruebas (pytest)` del *job* `verificar`.

In [ ]:
COPIA = u.copiar_repositorio(RAIZ, u.directorio_temporal())
PYTHON = u.crear_entorno_api(COPIA / "api", COPIA.parent)
proceso = u.ejecutar([str(PYTHON), "-m", "pytest"], COPIA / "api", comprobar=False)
print(proceso.stdout[-600:])
assert proceso.returncode == 0, proceso.stdout + proceso.stderr
assert (COPIA / "api" / "coverage.xml").exists()

La siguiente celda leerá `coverage.xml` con `xml.etree` y calculará el porcentaje de líneas cubiertas a partir del atributo `line-rate`. Lo comparará con el valor de `--cov-fail-under` de `pyproject.toml`.

In [ ]:
import re
import xml.etree.ElementTree as ET

raiz_xml = ET.parse(COPIA / "api" / "coverage.xml").getroot()
porcentaje = float(raiz_xml.attrib["line-rate"]) * 100
minimo = float(re.search(r"--cov-fail-under=(\d+)", (COPIA / "api" / "pyproject.toml").read_text()).group(1))
paquetes = [p.attrib["name"] for p in raiz_xml.iter("package")]
print(f"Cobertura: {porcentaje:.1f} % (mínimo configurado: {minimo:.0f} %). Paquetes: {paquetes}")
assert porcentaje >= minimo

El archivo es el mismo tipo de contenido que se descarga del artefacto: un *XML* con el atributo `line-rate` (fracción de líneas cubiertas) y un elemento `package` por cada paquete medido.

## Ejercicio 2. Comprobar la retención del artefacto

El texto del cuaderno indica que el periodo de retención se configura con `retention-days`. Leer en `ci.yml` cuántos días se conserva el artefacto y comprobar que el valor es válido.

La siguiente celda leerá `ci.yml`, buscará el paso que usa `actions/upload-artifact` y comprobará su nombre, su ruta, su condición `always()` y que `retention-days` esté entre 1 y 90 (el rango admitido para repositorios).

In [ ]:
wf = u.cargar_workflow(RAIZ / ".github" / "workflows" / "ci.yml")
subida = [p for p in u.pasos(wf, "verificar") if p.get("uses", "").startswith("actions/upload-artifact@")]
assert len(subida) == 1
parametros = subida[0]["with"]
print(subida[0]["name"], "->", parametros)
assert parametros["name"] == "cobertura-pytest"
assert parametros["path"] == "api/coverage.xml"
assert subida[0]["if"] == "always()"
assert 1 <= int(parametros["retention-days"]) <= 90

## Ejercicio 3. Localizar y descargar el artefacto

Al finalizar una ejecución, abrir el apartado `Artifacts` del resumen y descargar `cobertura-pytest`.

La siguiente celda listará los artefactos de la última ejecución del *workflow* `CI` con la *API* de *GitHub*, mostrará su nombre, tamaño y si caducaron, y descargará `cobertura-pytest` en una carpeta temporal.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
ID="$(gh run list --workflow CI --status completed --limit 1 --json databaseId --jq '.[0].databaseId')"
gh api "repos/${REPO}/actions/runs/${ID}/artifacts" \
  --jq '.artifacts[] | {name, size_in_bytes, expired}'
DESTINO="${TMPDIR:-/tmp}/descarga-cobertura"
gh run download "${ID}" --name cobertura-pytest --dir "${DESTINO}"
ls -l "${DESTINO}"

**Salida esperada.** Un objeto con `"name": "cobertura-pytest"` y `"expired": false`, y el archivo `coverage.xml` en la carpeta temporal. Se puede comprobar con el mismo código del ejercicio 1 apuntando a ese archivo.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>